# 15_Task8_Step3b — Ensemble nhiều bản Step 2 khác seed

**Mục tiêu:** train thêm 3 bản `task8_step2_distance` với seed 1, 2, 3 (cùng đặc trưng 546, cùng augmentation, cùng cấu hình với Step 2 gốc là seed 0), rồi trung bình xác suất cả 4 bản. So với `task8_step2_distance` (seed 0) xem có tốt hơn không.

- Train tối đa 100 epoch, early stopping 18 (theo Task 8), CUDA.
- Checkpoint mới lưu riêng ở `Data_321/checkpoints_v2/task8_step3b/seed{1,2,3}/TCN.pt`. **Không ghi đè** Step 2. Chạy lại thì tự nạp checkpoint có sẵn, không train lại (đặt `FORCE_RETRAIN = True` ở cell 4 nếu muốn train lại).
- Đánh giá giống 3a: tập test, xoay giả lập 0/15/30/45/60° (cùng seed xoay với 3a nên dòng Step 2 phải ra đúng số của 3a), tiêu chí ngưỡng 0.5→0.9, ms đoán đúng đầu tiên, độ chính xác mọi khung, độ chính xác khi tin cậy ≥ 0.7, độ phủ.
- Bảng cũng hiện từng seed riêng lẻ để biết kết quả của Step 2 có phải do "may mắn lúc train" không.
- **Tiêu chí dừng:** nếu ensemble không hơn Step 2 thì bỏ 3b, giữ Step 2.
- Cell 7–8: webcam, phím `m` đổi giữa ensemble 4 model và Step 2 gốc.

**Chạy lần lượt cell 1 → 8** bằng kernel `train-gpu` (có CUDA và MediaPipe).

In [9]:
#cell 1: import, nạp train/val/test, dựng đặc trưng 126 như notebook 13
import math, time, collections
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
assert (BASE / "manifest.csv").exists(), f"Không thấy {BASE / 'manifest.csv'}"
m = pd.read_csv(BASE / "manifest.csv")
Tmax = int(m.khung.max())
FPS_DATA = 120

def load_base_features(f):
    lm = np.load(BASE / f)["lm"]
    lm = lm - lm[:, :1]                                        # cổ tay làm gốc
    lm = lm / np.linalg.norm(lm[:, 9], axis=1)[:, None, None]  # chuẩn hóa cỡ bàn tay
    pos = lm.reshape(len(lm), 63)
    has = ~np.isnan(pos[:, 0])
    vel = np.vstack([np.zeros((1, 63), np.float32), np.diff(pos, axis=0)])
    return np.nan_to_num(np.hstack([pos, vel])).astype(np.float32), has

def build(df):
    X = np.zeros((len(df), Tmax, 126), np.float32)
    Mk = np.zeros((len(df), Tmax), bool)
    for i, f in enumerate(df.path):
        x, h = load_base_features(f)
        X[i, :len(x)], Mk[i, :len(x)] = x, h
    return (torch.tensor(X, device=dev),
            torch.tensor(df.nhan.values, dtype=torch.long, device=dev),
            torch.tensor(Mk, device=dev))

meta = {t: m[m.tap == t].reset_index(drop=True) for t in ["train", "val", "test"]}
data = {t: build(meta[t]) for t in meta}
Xte, yte, Mte = data["test"]
print("CUDA:", torch.cuda.get_device_name(0), "| Tmax:", Tmax)
for t, (X, y, Mk) in data.items():
    print(t, tuple(X.shape), "| video:", len(y), "| khung có tay:", int(Mk.sum().item()))

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (151, 355, 126) | video: 151 | khung có tay: 19108
val (31, 355, 126) | video: 31 | khung có tay: 3782
test (33, 355, 126) | video: 33 | khung có tay: 3930


In [10]:
#cell 2: đặc trưng hình học 546, augmentation, hàm loss, xoay giả lập, TCN (giống notebook 13)
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def features_from_pos(pos, hand_mask):
    valid = hand_mask[:, :, None, None]
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    pos_flat = pos.reshape(b, T, 63)
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(pos_flat)
    vel[:, 1:] = (pos_flat[:, 1:] - pos_flat[:, :-1]) * pair_valid
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist)
    dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_flat, vel, dist, dist_vel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

def augment_geo(X, hand_mask, rot_deg=(40, 45, 40), noise_std=0.005):
    b, T = X.shape[:2]
    pos = X[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    rx, ry, rz = [math.radians(d) for d in rot_deg]
    ax = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * rx
    ay = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * ry
    az = (torch.rand(b, 1, 1, device=X.device) * 2 - 1) * rz
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    flip = torch.rand(b, 1, 1, device=X.device) < 0.5
    pos[..., 0] = torch.where(flip, -pos[..., 0], pos[..., 0])
    pos = pos + torch.randn_like(pos) * noise_std * valid
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    return features_from_pos(pos, hand_mask)

def loss_acc(net, X, y, Mk):
    out = net(X)
    loss_each = F.cross_entropy(out.transpose(1, 2), y[:, None].expand(-1, X.shape[1]), reduction="none")
    correct = out.argmax(-1) == y[:, None]
    return (loss_each * Mk).sum() / Mk.sum().clamp(min=1), (correct & Mk).sum() / Mk.sum().clamp(min=1)

def rotate_base(X, hand_mask, ax_deg, ay_deg, az_deg):
    b, T = X.shape[:2]
    pos = X[..., :63].reshape(b, T, 21, 3).clone()
    valid = hand_mask[:, :, None, None]
    ax, ay, az = [torch.as_tensor(a, dtype=torch.float32, device=X.device) * math.pi / 180 for a in (ax_deg, ay_deg, az_deg)]
    xx, yy, zz = pos[..., 0], pos[..., 1], pos[..., 2]
    y1, z1 = yy * ax.cos() - zz * ax.sin(), yy * ax.sin() + zz * ax.cos()
    x2, z2 = xx * ay.cos() + z1 * ay.sin(), -xx * ay.sin() + z1 * ay.cos()
    pos = torch.stack([x2 * az.cos() - y1 * az.sin(), x2 * az.sin() + y1 * az.cos(), z2], dim=-1)
    pos = torch.where(valid, pos, torch.zeros_like(pos)).reshape(b, T, 63)
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(pos)
    vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * pair_valid
    return torch.cat([pos, vel], dim=-1)

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__()
        self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))
print("Đã định nghĩa đặc trưng, augmentation, loss, xoay giả lập và TCN.")

Đã định nghĩa đặc trưng, augmentation, loss, xoay giả lập và TCN.


In [11]:
#cell 3: cấu hình 3b, nạp Step 2 gốc (seed 0) làm mốc, dựng đặc trưng val 546 một lần
SEEDS = [1, 2, 3]
CKPT_STEP2 = BASE / "checkpoints_v2" / "task8_step2_distance" / "TCN.pt"
assert CKPT_STEP2.exists(), f"Không thấy checkpoint Step 2: {CKPT_STEP2}"
CFG = {"feature_version": "pairwise_distances_v1", "rot": (40, 45, 40),
       "noise": 0.005, "epochs": 100, "patience": 18, "in_features": 546}
ckpt_path = lambda s: BASE / "checkpoints_v2" / "task8_step3b" / f"seed{s}" / "TCN.pt"

ck2 = torch.load(CKPT_STEP2, map_location=dev, weights_only=False)
if ck2.get("cfg") != CFG:
    print("CẢNH BÁO: cấu hình Step 2 gốc khác CFG ở đây; so sánh có thể không công bằng:", ck2.get("cfg"))
net_step2 = TCN(in_features=546).to(dev)
net_step2.load_state_dict(ck2["w"])
net_step2.eval()

Xv, yv, Mv = data["val"]
geo_val = geo_from_base(Xv, Mv)
print(f"Step 2 gốc (seed 0): epoch {ck2['epoch']} | val_loss {ck2['val_loss']:.4f}")
print("Checkpoint 3b sẽ lưu ở:", BASE / "checkpoints_v2" / "task8_step3b")

Step 2 gốc (seed 0): epoch 25 | val_loss 0.2104
Checkpoint 3b sẽ lưu ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_v2\task8_step3b


In [12]:
#cell 4: train 3 bản seed 1,2,3 trên CUDA (tối đa 100 epoch, dừng sớm 18); có checkpoint thì tự nạp
FORCE_RETRAIN = False  # True chỉ khi muốn train lại cả 3 bản

def train_seed(seed, force=False, bs=16):
    cfg = {**CFG, "seed": seed}
    path = ckpt_path(seed)
    net = TCN(in_features=CFG["in_features"]).to(dev)
    if path.exists() and not force:
        ck = torch.load(path, map_location=dev, weights_only=False)
        if ck.get("cfg") == cfg:
            net.load_state_dict(ck["w"])
            print(f"seed {seed}: nạp checkpoint có sẵn | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")
            return net.eval()
        print(f"seed {seed}: checkpoint có cấu hình khác; sẽ train lại.")

    torch.manual_seed(seed)
    net = TCN(in_features=CFG["in_features"]).to(dev)   # khởi tạo lại sau khi đặt seed
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    Xt, yt, Mt = data["train"]
    best, wait, hist, t0 = float("inf"), 0, [], time.time()
    for ep in range(1, CFG["epochs"] + 1):
        net.train()
        perm = torch.randperm(len(Xt), device=dev)
        total = 0.0
        for s0 in range(0, len(Xt), bs):
            ids = perm[s0:s0 + bs]
            xb = augment_geo(Xt[ids], Mt[ids], CFG["rot"], CFG["noise"])
            loss, _ = loss_acc(net, xb, yt[ids], Mt[ids])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.item() * len(ids)
        net.eval()
        with torch.no_grad():
            vl, va = loss_acc(net, geo_val, yv, Mv)
        hist.append({"epoch": ep, "train_loss": total / len(Xt), "val_loss": float(vl.item()), "val_acc": float(va.item())})
        if hist[-1]["val_loss"] < best:
            best, wait = hist[-1]["val_loss"], 0
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({"w": net.state_dict(), "epoch": ep, "val_loss": best, "hist": hist,
                        "model": "TCN", "variant": f"task8_step3b_seed{seed}", "cfg": cfg}, path)
        else:
            wait += 1
        if ep % 10 == 0 or wait >= CFG["patience"] or ep == CFG["epochs"]:
            print(f"seed {seed} ep{ep:03d} | train {hist[-1]['train_loss']:.4f} | val {hist[-1]['val_loss']:.4f} | val_acc {hist[-1]['val_acc']:.3f}")
        if wait >= CFG["patience"]:
            print(f"seed {seed}: dừng sớm.")
            break
    ck = torch.load(path, map_location=dev, weights_only=False)
    net.load_state_dict(ck["w"])
    print(f"seed {seed}: tốt nhất epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f} | {time.time() - t0:.0f}s | đã lưu: {path}")
    return net.eval()

nets = {s: train_seed(s, FORCE_RETRAIN) for s in SEEDS}
ENS_NETS = [net_step2] + [nets[s] for s in SEEDS]
with torch.no_grad():
    for name, n in zip(["step2 (seed 0)"] + [f"seed {s}" for s in SEEDS], ENS_NETS):
        vl, va = loss_acc(n, geo_val, yv, Mv)
        print(f"VAL {name:15s} | loss {vl.item():.4f} | acc {va.item():.3f}")

seed 1: nạp checkpoint có sẵn | epoch 32 | val_loss 0.1936
seed 2: nạp checkpoint có sẵn | epoch 35 | val_loss 0.2167
seed 3: nạp checkpoint có sẵn | epoch 34 | val_loss 0.2024
VAL step2 (seed 0)  | loss 0.2104 | acc 0.923
VAL seed 1          | loss 0.1936 | acc 0.924
VAL seed 2          | loss 0.2167 | acc 0.909
VAL seed 3          | loss 0.2024 | acc 0.926


In [13]:
#cell 5: chấm điểm theo tiêu chí Decision; so step2, từng seed riêng lẻ và ensemble 4 model ở góc 0/15/30/45/60
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]

def metrics(P, y, Mk):
    conf, pred = P.max(-1)
    ok = (pred == y[:, None]) & Mk
    n = Mk.sum().item()
    r = {"acc_all": ok.sum().item() / n * 100}
    for t in THRS:
        k = int(round(t * 100))
        hit = ok & (conf >= t)
        has = hit.any(1)
        first = hit.float().argmax(1).float()
        r[f"v{k}"] = has.float().mean().item() * 100
        r[f"ms{k}"] = (first[has].mean().item() / FPS_DATA * 1000) if has.any() else float("nan")
    r["tb"] = float(np.mean([r[f"v{int(round(t*100))}"] for t in THRS]))
    sel = (conf >= 0.7) & Mk
    r["accc70"] = (ok & sel).sum().item() / max(sel.sum().item(), 1) * 100
    r["cov70"] = sel.sum().item() / n * 100
    return r

NAMES = ["step2 (seed 0)"] + [f"seed {s}" for s in SEEDS] + ["ensemble 3b (4 model)"]
MAGS, REPEAT = [0, 15, 30, 45, 60], 5
g = torch.Generator(device=dev).manual_seed(0)   # cùng seed xoay với 3a
rows = []
with torch.no_grad():
    for mag in MAGS:
        acc = {n: [] for n in NAMES}
        for _ in range(1 if mag == 0 else REPEAT):
            if mag == 0:
                Xr = Xte
            else:
                ang = lambda: (torch.rand(len(Xte), 1, 1, device=dev, generator=g) * 2 - 1) * mag
                Xr = rotate_base(Xte, Mte, ang(), ang(), ang())
            geo = geo_from_base(Xr, Mte)
            Ps = [n(geo).softmax(-1) for n in ENS_NETS]
            for name, P in zip(NAMES, Ps + [torch.stack(Ps).mean(0)]):
                acc[name].append(metrics(P, yte, Mte))
        for name, v in acc.items():
            rows.append({"model": name, "goc": mag, **{k: float(np.nanmean([d[k] for d in v])) for k in v[0]}})
res = pd.DataFrame(rows)

def show(col, title):
    print("\n" + title)
    display(res.pivot_table(index="model", columns="goc", values=col).reindex(NAMES).round(1))

show("tb",      "Tiêu chí Decision: TB 5 ngưỡng của tỉ lệ video (%)")
show("v50",     "Tỉ lệ video ở ngưỡng 0.5 (%)")
show("v60",     "Tỉ lệ video ở ngưỡng 0.6 (%)")
show("acc_all", "Độ chính xác mọi khung có tay (%)")
show("accc70",  "Độ chính xác khi tin cậy ≥ 0.7 (%)")
show("cov70",   "Độ phủ: % khung đạt ngưỡng 0.7")
show("ms70",    "Thời điểm đoán đúng đầu tiên ở ngưỡng 0.7 (ms, tính từ đầu video)")


Tiêu chí Decision: TB 5 ngưỡng của tỉ lệ video (%)


goc,0,15,30,45,60
model,,,,,
step2 (seed 0),98.2,98.5,98.7,98.7,98.8
seed 1,98.8,98.8,98.8,98.8,98.9
seed 2,98.8,98.8,99.0,98.9,98.9
seed 3,98.2,98.2,98.2,98.3,98.4
ensemble 3b (4 model),98.2,98.2,98.5,98.4,98.7



Tỉ lệ video ở ngưỡng 0.5 (%)


goc,0,15,30,45,60
model,,,,,
step2 (seed 0),100.0,100.0,100.0,100.0,100.0
seed 1,100.0,100.0,100.0,100.0,100.0
seed 2,100.0,100.0,100.0,100.0,100.0
seed 3,100.0,100.0,100.0,100.0,100.0
ensemble 3b (4 model),100.0,100.0,100.0,100.0,100.0



Tỉ lệ video ở ngưỡng 0.6 (%)


...

goc,0,15,30,45,60
model,,,,,
step2 (seed 0),95.8,95.9,95.9,95.3,95.6
seed 1,98.7,98.7,98.4,98.0,97.3
seed 2,97.8,97.6,97.5,96.8,97.1
seed 3,97.1,97.0,96.8,96.0,96.4
ensemble 3b (4 model),97.5,97.4,97.4,96.7,96.7



Độ phủ: % khung đạt ngưỡng 0.7


goc,0,15,30,45,60
model,,,,,
step2 (seed 0),91.8,91.7,91.7,92.1,91.8
seed 1,88.6,88.6,88.8,89.5,89.7
seed 2,88.5,88.8,88.8,89.7,89.5
seed 3,87.9,88.1,88.8,90.0,89.7
ensemble 3b (4 model),89.5,89.5,89.5,90.4,90.2



Thời điểm đoán đúng đầu tiên ở ngưỡng 0.7 (ms, tính từ đầu video)


goc,0,15,30,45,60
model,,,,,
step2 (seed 0),116.4,116.1,116.8,117.5,114.4
seed 1,117.7,118.5,119.5,113.4,111.0
seed 2,121.0,121.3,123.7,122.8,124.4
seed 3,109.8,113.8,115.7,111.6,114.4
ensemble 3b (4 model),119.4,120.1,121.0,119.2,117.9


In [14]:
#cell 6: chênh lệch ensemble so với step2 gốc, độ lệch giữa các seed, kết luận sơ bộ
E, S = "ensemble 3b (4 model)", "step2 (seed 0)"
p = res.pivot_table(index="model", columns="goc", values=["tb", "v50", "v60", "acc_all", "accc70", "cov70"])
for col in ["tb", "v50", "v60", "acc_all", "accc70", "cov70"]:
    d = p[col].loc[E] - p[col].loc[S]
    print(f"Δ {col:8s} (ensemble - step2): " + " | ".join(f"{int(a):2d}°: {v:+.1f}" for a, v in d.items()))

single = p["acc_all"].loc[[S] + [f"seed {s}" for s in SEEDS]]
print("\nacc_all của 4 model đơn lẻ: nhỏ nhất → lớn nhất")
print(" | ".join(f"{int(a):2d}°: {single[a].min():.1f}→{single[a].max():.1f}" for a in single.columns))

d_acc = p["acc_all"].loc[E] - p["acc_all"].loc[S]
if d_acc[0] >= -1 and d_acc[45] > 0:
    print("\nKẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ (không giảm quá 1 điểm ở 0°, tăng ở ±45°). Cần xác nhận bằng webcam và FPS.")
else:
    print("\nKẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; bỏ 3b, giữ step2 gốc.")
print("Lưu ý: tiêu chí chỉ để tham khảo; test chỉ 33 video, chênh dưới ~1 điểm hoặc nhỏ hơn độ lệch giữa các seed thì chưa chắc chắn.")

Δ tb       (ensemble - step2):  0°: +0.0 | 15°: -0.4 | 30°: -0.1 | 45°: -0.2 | 60°: -0.1
Δ v50      (ensemble - step2):  0°: +0.0 | 15°: +0.0 | 30°: +0.0 | 45°: +0.0 | 60°: +0.0
Δ v60      (ensemble - step2):  0°: +0.0 | 15°: +0.0 | 30°: +0.0 | 45°: +0.0 | 60°: +0.0
Δ acc_all  (ensemble - step2):  0°: +0.6 | 15°: +1.2 | 30°: +1.0 | 45°: +1.0 | 60°: +1.2
Δ accc70   (ensemble - step2):  0°: +1.7 | 15°: +1.5 | 30°: +1.5 | 45°: +1.5 | 60°: +1.1
Δ cov70    (ensemble - step2):  0°: -2.3 | 15°: -2.1 | 30°: -2.2 | 45°: -1.8 | 60°: -1.6

acc_all của 4 model đơn lẻ: nhỏ nhất → lớn nhất
 0°: 92.6→94.4 | 15°: 92.7→94.6 | 30°: 92.6→94.2 | 45°: 92.8→94.0 | 60°: 92.3→93.6

KẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ (không giảm quá 1 điểm ở 0°, tăng ở ±45°). Cần xác nhận bằng webcam và FPS.
Lưu ý: tiêu chí chỉ để tham khảo; test chỉ 33 video, chênh dưới ~1 điểm hoặc nhỏ hơn độ lệch giữa các seed thì chưa chắc chắn.


In [15]:
#cell 7: chuẩn bị MediaPipe + dự đoán webcam cho 2 chế độ: ensemble 4 model và step2 gốc
import cv2
import mediapipe as mp

TASK = BASE / "hand_landmarker.task"
assert TASK.exists(), f"Không thấy {TASK}"
CLASSES = ["paper", "rock", "scissors"]
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}
GRAY, BLACK = (190, 190, 190), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
MODES = ["ens", "new"]
MODE_NAME = {"ens": "ENSEMBLE 3b (4 model)", "new": "STEP 2 (seed 0)"}

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(
    mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not result.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in result.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def webcam_base_features(lm):
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    scale = np.linalg.norm(q[:, 9], axis=1)[:, None, None]
    q = np.nan_to_num(q / np.maximum(scale, 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def predict(which):
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = webcam_base_features(lm)
    xb, mb = torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        geo = geo_from_base(xb, mb)                      # tính đặc trưng 546 một lần cho mọi model
        nets_use = ENS_NETS if which == "ens" else [net_step2]
        probs = torch.stack([n(geo).softmax(-1)[0, -1] for n in nets_use]).mean(0)
    conf, cls = probs.max(0)
    return int(cls.item()), float(conf.item())

def put_label(img, text, x, y, bg, scale=0.75):
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y-h-base-6), (x+w+10, y+2), bg, -1)
    cv2.putText(img, text, (x+5, y-base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, landmarks, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (landmarks[:, 0] * w).astype(int), (landmarks[:, 1] * h).astype(int)
    if mirror:
        px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2)
    cv2.rectangle(img, (max(0, px.min()-12), max(0, py.min()-12)),
                  (min(w-1, px.max()+12), min(h-1, py.max()+12)), color, 2)

def draw_hud(img, fps, threshold, which):
    put_label(img, f"FPS {fps:.1f} | threshold {threshold:.2f}", 10, 30, (230, 230, 230), 0.6)
    put_label(img, f"MODEL: {MODE_NAME[which]}", 10, 65, (230, 230, 230), 0.6)
    put_label(img, "m: doi model | [ ]: nguong | q: thoat", 10, 100, (230, 230, 230), 0.55)
print("Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.")

Đã nạp MediaPipe; webcam chỉ chạy ở cell 8.


In [17]:
#cell 8: chạy webcam; m đổi model (ensemble 4 ↔ step2), [ ] chỉnh ngưỡng, q thoát; nhìn FPS để biết ensemble có chậm không
CAM_ID, MIRROR, THR_CAM, WHICH = 0, True, 0.70, "ens"
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam; thử đổi CAM_ID = 1."

buf.clear()
state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        out = predict(WHICH) if lm is not None else None
        disp = cv2.flip(frame, 1) if MIRROR else frame
        fps = 0.9 * fps + 0.1 / max(now - last, 1e-6)
        last = now
        if out is None:
            put_label(disp, "NO HAND", 10, 150, GRAY)
        else:
            cls, conf = out
            sure = conf >= THR_CAM
            color = COLORS[cls] if sure else GRAY
            draw_hand(disp, lm, color, MIRROR)
            put_label(disp, f"{CLASSES[cls]} {conf:.0%}" if sure else f"{CLASSES[cls]}? {conf:.0%}", 10, 150, color)
        draw_hud(disp, fps, THR_CAM, WHICH)
        cv2.imshow("Task 8 Step 3b - RPS webcam", disp)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("m"):
            WHICH = MODES[(MODES.index(WHICH) + 1) % len(MODES)]
        elif key == ord("["):
            THR_CAM = max(0.05, THR_CAM - 0.05)
        elif key == ord("]"):
            THR_CAM = min(0.95, THR_CAM + 0.05)
finally:
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()
print("Đã đóng webcam.")

Đã đóng webcam.
